# PCN Detection — Evaluation Corpus Generator

Generates a labelled test corpus with ground-truth annotations for evaluating the PCN detection pipeline.

**Test categories:**
- **True positives (tp):** valid Luhn cards across 7 types × 3 separator styles
- **Hard negatives (hn):** phone numbers, account numbers, random Luhn-failing digit sequences
- **Edge cases (ec):** boundary positions, multiple PCNs per field, adjacent numbers, out-of-range Luhn-valid numbers

Each row carries a JSON `annotations` column with span-level ground truth for automated evaluation.

In [0]:
import json
import random
import pandas as pd
from pcn_utils import (
    CARD_TYPES, SEPARATORS,
    generate_card_number, generate_luhn_failing_number,
    generate_luhn_valid_number, passes_luhn,
)

EVAL_DATABASE = "capstone_project.pii_data"
EVAL_TABLE = "pcn_evaluation_corpus"
random.seed(42)

NAMES = [
    "Alice Chen", "Bob Martinez", "Carol Williams", "David Kim",
    "Eva Thompson", "Frank Garcia", "Grace Lee", "Henry Patel",
    "Iris Johnson", "James Wilson", "Karen Brown", "Leo Davis",
    "Maria Rodriguez", "Nathan Moore", "Olivia Taylor", "Peter White",
]

rows = []

---
## True Positives

For each card type × separator style, generate rows with:
- `card_number_field`: the card number alone
- `text_field`: card number embedded in a transaction-like string
- Annotations marking both occurrences

In [0]:
ROWS_PER_COMBO = 10

for card_type in CARD_TYPES:
    for sep_name in SEPARATORS:
        for i in range(ROWS_PER_COMBO):
            formatted, digits, ct = generate_card_number(card_type, sep_name)
            name = random.choice(NAMES)
            roc = random.randint(100000, 999999)

            prefix_text = f"/ROC/{roc}/URI/CC PAYMENT FOR {name.upper()} "
            text_field = prefix_text + formatted
            text_start = len(prefix_text)
            text_end = text_start + len(formatted)

            annotations = [
                {
                    "field": "card_number_field",
                    "start": 0,
                    "end": len(formatted),
                    "raw_pcn": formatted,
                    "digits": digits,
                    "card_type": ct,
                    "separator": sep_name,
                },
                {
                    "field": "text_field",
                    "start": text_start,
                    "end": text_end,
                    "raw_pcn": formatted,
                    "digits": digits,
                    "card_type": ct,
                    "separator": sep_name,
                },
            ]

            rows.append({
                "test_id": f"tp_{ct.lower()}_{sep_name}_{i:03d}",
                "test_category": "tp",
                "test_label": f"{ct.lower()}_{sep_name}",
                "text_field": text_field,
                "card_number_field": formatted,
                "other_number_field": None,
                "metadata_field": name,
                "annotations": json.dumps(annotations),
            })

print(f"True positives: {len(rows)} rows ({len(CARD_TYPES)} types × {len(SEPARATORS)} separators × {ROWS_PER_COMBO} each)")

---
## Hard Negatives

Numeric sequences that should NOT be masked: phone numbers, account numbers, and random 13–19 digit strings that fail Luhn.

In [0]:
hn_start = len(rows)

# --- Phone numbers (10-11 digits, various formats) ---
PHONE_FORMATS = [
    lambda: f"+1-{random.randint(200,999)}-{random.randint(200,999)}-{random.randint(1000,9999)}",
    lambda: f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}",
    lambda: f"{random.randint(200,999)}.{random.randint(200,999)}.{random.randint(1000,9999)}",
    lambda: f"{random.randint(200,999)}{random.randint(200,999)}{random.randint(1000,9999)}",
]

for i in range(20):
    phone = random.choice(PHONE_FORMATS)()
    name = random.choice(NAMES)
    rows.append({
        "test_id": f"hn_phone_{i:03d}",
        "test_category": "hn",
        "test_label": "phone",
        "text_field": f"Contact {name} at {phone} for details",
        "card_number_field": None,
        "other_number_field": phone,
        "metadata_field": name,
        "annotations": json.dumps([]),
    })

# --- Account numbers (10-12 digits) ---
for i in range(20):
    length = random.randint(10, 12)
    acct = "".join(str(random.randint(0, 9)) for _ in range(length))
    name = random.choice(NAMES)
    rows.append({
        "test_id": f"hn_account_{i:03d}",
        "test_category": "hn",
        "test_label": "account_number",
        "text_field": f"Account {acct} for {name}",
        "card_number_field": None,
        "other_number_field": acct,
        "metadata_field": name,
        "annotations": json.dumps([]),
    })

# --- Random 13-19 digit sequences that fail Luhn ---
for i in range(40):
    length = random.randint(13, 19)
    number = generate_luhn_failing_number(length)
    sep_choice = random.choice(["", " ", "-"])
    if sep_choice and length == 16:
        display_num = sep_choice.join([number[j:j+4] for j in range(0, 16, 4)])
    else:
        display_num = number
    name = random.choice(NAMES)
    rows.append({
        "test_id": f"hn_luhn_fail_{i:03d}",
        "test_category": "hn",
        "test_label": f"random_{length}d_luhn_fail",
        "text_field": f"Reference number {display_num} for {name}",
        "card_number_field": None,
        "other_number_field": display_num,
        "metadata_field": name,
        "annotations": json.dumps([]),
    })

# --- SSNs (9 digits, well below range but verify no match) ---
for i in range(10):
    area = random.randint(100, 899)
    group = random.randint(10, 99)
    serial = random.randint(1000, 9999)
    ssn = f"{area}-{group}-{serial}"
    rows.append({
        "test_id": f"hn_ssn_{i:03d}",
        "test_category": "hn",
        "test_label": "ssn",
        "text_field": f"SSN: {ssn}",
        "card_number_field": None,
        "other_number_field": ssn,
        "metadata_field": random.choice(NAMES),
        "annotations": json.dumps([]),
    })

print(f"Hard negatives: {len(rows) - hn_start} rows")

---
## Edge Cases

Boundary conditions: PCN at start/end of string, multiple PCNs in one field, PCN adjacent to other numbers, and out-of-range Luhn-valid numbers (12 and 20 digits) that must NOT be masked.

In [0]:
ec_start = len(rows)

# --- PCN at very start of text_field ---
for i in range(10):
    formatted, digits, ct = generate_card_number(separator_name=random.choice(list(SEPARATORS)))
    suffix = f" payment from {random.choice(NAMES)}"
    text_field = formatted + suffix
    rows.append({
        "test_id": f"ec_start_{i:03d}",
        "test_category": "ec",
        "test_label": "pcn_at_start",
        "text_field": text_field,
        "card_number_field": formatted,
        "other_number_field": None,
        "metadata_field": None,
        "annotations": json.dumps([
            {"field": "card_number_field", "start": 0, "end": len(formatted),
             "raw_pcn": formatted, "digits": digits, "card_type": ct, "separator": "mixed"},
            {"field": "text_field", "start": 0, "end": len(formatted),
             "raw_pcn": formatted, "digits": digits, "card_type": ct, "separator": "mixed"},
        ]),
    })

# --- PCN at very end of text_field ---
for i in range(10):
    formatted, digits, ct = generate_card_number(separator_name=random.choice(list(SEPARATORS)))
    prefix_text = f"Payment by {random.choice(NAMES)} card "
    text_field = prefix_text + formatted
    rows.append({
        "test_id": f"ec_end_{i:03d}",
        "test_category": "ec",
        "test_label": "pcn_at_end",
        "text_field": text_field,
        "card_number_field": formatted,
        "other_number_field": None,
        "metadata_field": None,
        "annotations": json.dumps([
            {"field": "card_number_field", "start": 0, "end": len(formatted),
             "raw_pcn": formatted, "digits": digits, "card_type": ct, "separator": "mixed"},
            {"field": "text_field", "start": len(prefix_text), "end": len(text_field),
             "raw_pcn": formatted, "digits": digits, "card_type": ct, "separator": "mixed"},
        ]),
    })

# --- Multiple PCNs in one text_field ---
for i in range(10):
    f1, d1, ct1 = generate_card_number(separator_name="hyphen")
    f2, d2, ct2 = generate_card_number(separator_name="space")
    mid = " and also "
    text_field = f"Cards {f1}{mid}{f2} on file"
    s1 = len("Cards ")
    e1 = s1 + len(f1)
    s2 = e1 + len(mid)
    e2 = s2 + len(f2)
    rows.append({
        "test_id": f"ec_multi_{i:03d}",
        "test_category": "ec",
        "test_label": "multi_pcn",
        "text_field": text_field,
        "card_number_field": f1,
        "other_number_field": None,
        "metadata_field": None,
        "annotations": json.dumps([
            {"field": "card_number_field", "start": 0, "end": len(f1),
             "raw_pcn": f1, "digits": d1, "card_type": ct1, "separator": "hyphen"},
            {"field": "text_field", "start": s1, "end": e1,
             "raw_pcn": f1, "digits": d1, "card_type": ct1, "separator": "hyphen"},
            {"field": "text_field", "start": s2, "end": e2,
             "raw_pcn": f2, "digits": d2, "card_type": ct2, "separator": "space"},
        ]),
    })

# --- PCN adjacent to a non-PCN number ---
for i in range(10):
    acct = "".join(str(random.randint(0, 9)) for _ in range(12))
    formatted, digits, ct = generate_card_number(separator_name="none")
    text_field = f"Acct {acct} Card {formatted} end"
    card_start = len(f"Acct {acct} Card ")
    card_end = card_start + len(formatted)
    rows.append({
        "test_id": f"ec_adjacent_{i:03d}",
        "test_category": "ec",
        "test_label": "pcn_adjacent_number",
        "text_field": text_field,
        "card_number_field": formatted,
        "other_number_field": acct,
        "metadata_field": None,
        "annotations": json.dumps([
            {"field": "card_number_field", "start": 0, "end": len(formatted),
             "raw_pcn": formatted, "digits": digits, "card_type": ct, "separator": "none"},
            {"field": "text_field", "start": card_start, "end": card_end,
             "raw_pcn": formatted, "digits": digits, "card_type": ct, "separator": "none"},
        ]),
    })

# --- 12-digit Luhn-valid (too short — must NOT be masked) ---
for i in range(10):
    number = generate_luhn_valid_number(12)
    rows.append({
        "test_id": f"ec_12digit_{i:03d}",
        "test_category": "ec",
        "test_label": "12digit_luhn_valid",
        "text_field": f"Short ref {number} here",
        "card_number_field": None,
        "other_number_field": number,
        "metadata_field": None,
        "annotations": json.dumps([]),
    })

# --- 20-digit Luhn-valid (too long — must NOT be masked) ---
for i in range(10):
    number = generate_luhn_valid_number(20)
    rows.append({
        "test_id": f"ec_20digit_{i:03d}",
        "test_category": "ec",
        "test_label": "20digit_luhn_valid",
        "text_field": f"Long ref {number} here",
        "card_number_field": None,
        "other_number_field": number,
        "metadata_field": None,
        "annotations": json.dumps([]),
    })

print(f"Edge cases: {len(rows) - ec_start} rows")

---
## Self-Validation & Write to Delta

In [0]:
# Validate every annotation's span offsets and Luhn correctness
errors = []
for row in rows:
    annots = json.loads(row["annotations"])
    for a in annots:
        field_val = row.get(a["field"])
        if field_val is None:
            errors.append(f"{row['test_id']}: field '{a['field']}' is None")
            continue
        extracted = field_val[a["start"]:a["end"]]
        if extracted != a["raw_pcn"]:
            errors.append(
                f"{row['test_id']}: span mismatch in {a['field']} — "
                f"expected '{a['raw_pcn']}', got '{extracted}'"
            )
        if not passes_luhn(a["digits"]):
            errors.append(f"{row['test_id']}: Luhn FAIL on {a['digits']}")

if errors:
    for e in errors[:20]:
        print(f"  ERROR: {e}")
    raise ValueError(f"{len(errors)} annotation errors found")

print(f"Self-validation passed — all {sum(len(json.loads(r['annotations'])) for r in rows)} annotations verified.")

In [0]:
df_pandas = pd.DataFrame(rows)

# Summary
print(f"Total corpus: {len(rows)} rows")
print(f"\nBy category:")
print(df_pandas["test_category"].value_counts().to_string())
print(f"\nBy label:")
print(df_pandas["test_label"].value_counts().to_string())

# Write to Delta
qualified_name = f"{EVAL_DATABASE}.{EVAL_TABLE}"
print(f"\nWriting to '{qualified_name}' ...")

(
    spark.createDataFrame(df_pandas)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qualified_name)
)

print(f"✔ Corpus written to '{qualified_name}'.")
display(spark.table(qualified_name).limit(5))